# Organizational Graph Pipeline - Test Notebook

This notebook tests the integrated pipeline that:
1. **Loads data** from Excel files (organizational units + activities)
2. **Vectorizes** all documents in a vector store (ChromaDB or Mock)
3. **Builds graph** from vectorized documents in the vector index
4. **Uploads** to graph store (NetworkX local or FalkorDB cloud)

## Architecture
```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   DataSource    │ ──▶ │   VectorStore   │ ──▶ │  GraphBuilder   │ ──▶ │   GraphStore    │
│    (Excel)      │     │ (ChromaDB/Mock) │     │  (from vectors) │     │ (NX/FalkorDB)   │
└─────────────────┘     └─────────────────┘     └─────────────────┘     └─────────────────┘
```

## Key Abstractions
| Abstraction | Implementations | Purpose |
|-------------|-----------------|--------|
| `VectorStoreBase` | MockVectorStore, ChromaVectorStore | Embedding storage & similarity search |
| `GraphStoreBase` | NetworkXGraphStore, FalkorDBGraphStore | Graph database operations |
| `DataSourceBase` | ExcelDataSource | Load organizational data |
| `OrgGraphPipeline` | - | Orchestrates the full workflow |

In [14]:
# =============================================================================
# CONFIGURATION
# =============================================================================

import os
import sys
from dotenv import load_dotenv

sys.path.insert(0, '.')

# Load .env file (secrets and connection strings)
load_dotenv()

# Anthropic API key — from system environment variable only, not .env
#ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY must be set as a system environment variable. "
        "Set it in PyCharm Run Configuration or your shell profile."
    )

# ═══════════════════════════════════════════════════════════════════════════
# DATA SOURCE (from .env)
# ═══════════════════════════════════════════════════════════════════════════

DATA_SOURCE_PATH = os.getenv("DATA_SOURCE_PATH", "./data/samples/CASP_Activities_EN.xlsx")

# ═══════════════════════════════════════════════════════════════════════════
# GRAPH STRUCTURE OPTIONS (code defaults — not secrets)
# ═══════════════════════════════════════════════════════════════════════════

ACTIVITIES_AS_NODES = True
ACTIVITY_ID_PREFIX = "ACT"
MASTER_ROOT_ID = "ORG_MASTER"
MASTER_ROOT_NAME = "European Parliament Organizations"

# ═══════════════════════════════════════════════════════════════════════════
# VECTOR STORE OPTIONS (from .env)
# ═══════════════════════════════════════════════════════════════════════════

VECTOR_STORE_TYPE = os.getenv("VECTOR_STORE_TYPE", "mock")
VECTOR_PERSIST_DIR = os.getenv("VECTOR_PERSIST_DIR", "./chroma_db")
VECTOR_COLLECTION_NAME = os.getenv("VECTOR_COLLECTION_NAME", "org_documents")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")

# ═══════════════════════════════════════════════════════════════════════════
# GRAPH STORE OPTIONS (from .env — contains credentials)
# ═══════════════════════════════════════════════════════════════════════════

GRAPH_STORE_TYPE = os.getenv("GRAPH_STORE_TYPE", "networkx")
GRAPH_LOCAL_PATH = os.getenv("GRAPH_LOCAL_PATH", "./graph_data/organization_graph.json")

USE_FALKORDB = os.getenv("USE_FALKORDB", "false").lower() == "true"
FALKORDB_URL = os.getenv("FALKORDB_URL", "")
FALKORDB_GRAPH_NAME = os.getenv("FALKORDB_GRAPH_NAME", "org_hierarchy")

if USE_FALKORDB and not FALKORDB_URL:
    raise EnvironmentError(
        "FALKORDB_URL must be set in .env when USE_FALKORDB is enabled."
    )

# ═══════════════════════════════════════════════════════════════════════════
# PIPELINE FLAGS (code defaults — not secrets)
# ═══════════════════════════════════════════════════════════════════════════

CLEAR_EXISTING = True
CREATE_VECTOR_INDEX = True
SAVE_LOCAL = True
VERBOSE = True

# ═══════════════════════════════════════════════════════════════════════════
# DISPLAY OPTIONS
# ═══════════════════════════════════════════════════════════════════════════

SHOW_ACTIVITIES_IN_TREE = False

# ═══════════════════════════════════════════════════════════════════════════
# STARTUP SUMMARY (masks secrets)
# ═══════════════════════════════════════════════════════════════════════════

print("Configuration:")
print(f"  Data source:        {DATA_SOURCE_PATH}")
print(f"  Activities as nodes:{ACTIVITIES_AS_NODES}")
print(f"  Vector store:       {VECTOR_STORE_TYPE}")
print(f"  Embedding model:    {EMBEDDING_MODEL}")
print(f"  Graph store:        {GRAPH_STORE_TYPE}")
print(f"  Use FalkorDB:       {USE_FALKORDB}")
if USE_FALKORDB:
    # Show host only, mask credentials
    masked = FALKORDB_URL.split("@")[-1] if "@" in FALKORDB_URL else "(set)"
    print(f"  FalkorDB host:      {masked}")
print(f"  Anthropic API key:  {'...'+ANTHROPIC_API_KEY[-4:]}")

Configuration:
  Data source:        ./data/samples/CASP_Activities_EN.xlsx
  Activities as nodes:True
  Vector store:       mock
  Embedding model:    all-MiniLM-L6-v2
  Graph store:        networkx
  Use FalkorDB:       False
  Anthropic API key:  ...OAAA


---
## Step 1: Create Pipeline

In [2]:
# =============================================================================
# CREATE PIPELINE
# =============================================================================

from pipeline import create_pipeline, PipelineConfig, OrgGraphPipeline

# Create pipeline configuration
config = PipelineConfig(
    # Data source
    data_source_type="excel",
    data_source_path=DATA_SOURCE_PATH,
    activities_as_nodes=ACTIVITIES_AS_NODES,
    activity_id_prefix=ACTIVITY_ID_PREFIX,
    
    # Vector store
    vector_store_type=VECTOR_STORE_TYPE,
    vector_collection_name=VECTOR_COLLECTION_NAME,
    vector_persist_dir=VECTOR_PERSIST_DIR,
    embedding_model=EMBEDDING_MODEL,
    
    # Graph store
    graph_store_type="falkordb" if USE_FALKORDB else GRAPH_STORE_TYPE,
    graph_local_path=GRAPH_LOCAL_PATH,
    graph_name=FALKORDB_GRAPH_NAME,
    master_root_id=MASTER_ROOT_ID,
    master_root_name=MASTER_ROOT_NAME,
    falkordb_url=FALKORDB_URL if USE_FALKORDB else None,
    
    # Flags
    clear_existing=CLEAR_EXISTING,
    create_vector_index=CREATE_VECTOR_INDEX,
    save_local=SAVE_LOCAL,
    verbose=VERBOSE
)

# Create pipeline
pipeline = OrgGraphPipeline(config)

print("\n✓ Pipeline created")


✓ Pipeline created


---
## Step 2: Run Pipeline

In [7]:
# =============================================================================
# ORG HIERARCHY PARSER
# =============================================================================
import re

def parse_org_hierarchy(entity_codes):
    """
    Parse EP entity codes into parent-child relationships.
    
    Patterns:
      '22'       → top-level DG (root)
      '22-10'    → direct unit under DG (dash = attached unit)
      '22A'      → directorate under DG
      '22A10'    → unit under directorate 22A
      '22B0040'  → sub-unit under 22B00 (implicit parent synthesized)
    """
    codes = sorted(set(entity_codes))
    hierarchy = {}

    for code in codes:
        if re.match(r'^\d{2}$', code):
            hierarchy[code] = None
        elif re.match(r'^(\d{2})-\d+$', code):
            dg = re.match(r'^(\d{2})', code).group(1)
            hierarchy[code] = dg
        elif re.match(r'^\d{2}[A-Z]$', code):
            hierarchy[code] = code[:2]
        elif re.match(r'^\d{2}[A-Z]\d{2}$', code):
            hierarchy[code] = code[:3]
        elif re.match(r'^\d{2}[A-Z]\d{4,}$', code):
            directorate = code[:3]
            parent_code = code[:5]
            if parent_code not in codes and parent_code not in hierarchy:
                hierarchy[parent_code] = directorate
            hierarchy[code] = parent_code
        else:
            dg = re.match(r'^(\d{2})', code).group(1)
            hierarchy[code] = dg
            print(f"  ⚠️ Unrecognized code pattern '{code}', attached to root {dg}")

    return hierarchy

In [8]:
# =============================================================================
# PIPELINE STEP FUNCTIONS
# =============================================================================
import pandas as pd
import networkx as nx


def step_load_data(filepath):
    """Step 1: Load data — each row is one activity."""
    print("=" * 60)
    print("STEP 1: LOAD DATA")
    print("=" * 60)

    df = pd.read_excel(filepath)

    # Row number in Excel (1-based, header is row 1, data starts row 2)
    df['Excel_Row'] = range(1, len(df) + 1)

    # Synthetic activity ID: ORG_UNIT_ID.ROW_NUMBER
    df['Activity_ID'] = df['Entity Code'] + '.' + df['Excel_Row'].astype(str)

    print(f"  Total activities (rows): {len(df)}")
    print(f"  Org units: {df['Entity Code'].nunique()}")
    print(f"  Sample activity IDs:")
    for _, row in df.head(5).iterrows():
        print(f"    {row['Activity_ID']:15s} ({row['Entity Code']})")

    display(df[['Activity_ID', 'Entity Code', 
                 'Key Activities (Missions principales)', '%']].head(5))

    return df


def step_build_org_graph(df):
    """Step 2: Build org hierarchy + one node per activity."""
    print("=" * 60)
    print("STEP 2: BUILD ORG HIERARCHY + ACTIVITY NODES")
    print("=" * 60)

    G = nx.DiGraph()

    # --- Org unit nodes + hierarchy edges ---
    entity_codes = df['Entity Code'].unique()
    code_to_name = dict(zip(df['Entity Code'], df['Entity Name']))
    hierarchy = parse_org_hierarchy(entity_codes)

    for code, parent in hierarchy.items():
        G.add_node(code,
                    type='org_unit',
                    name=code_to_name.get(code, f'(synthesized) {code}'))
        if parent is not None:
            G.add_edge(parent, code, relation='HAS_CHILD')

    org_count = G.number_of_nodes()

    # --- Activity nodes: one per Excel row ---
    for _, row in df.iterrows():
        activity_id = row['Activity_ID']
        entity_code = row['Entity Code']

        G.add_node(activity_id,
                    type='activity',
                    description=row['Key Activities (Missions principales)'],
                    weight_pct=row['%'],
                    parent_org=entity_code,
                    excel_row=row['Excel_Row'])

        G.add_edge(entity_code, activity_id, relation='HAS_ACTIVITY')

    activity_count = G.number_of_nodes() - org_count

    print(f"  Org unit nodes:    {org_count}")
    print(f"  Activity nodes:    {activity_count}")
    print(f"  Total nodes:       {G.number_of_nodes()}")
    print(f"  Total edges:       {G.number_of_edges()}")

    synthesized = [n for n, d in G.nodes(data=True)
                   if d.get('type') == 'org_unit' and '(synthesized)' in d.get('name', '')]
    if synthesized:
        print(f"  Synthesized parents: {synthesized}")

    return G


def step_vectorize(pipeline, df):
    """Step 3: Vectorize activity descriptions."""
    print("=" * 60)
    print("STEP 3: VECTORIZE ACTIVITIES")
    print("=" * 60)

    documents = []
    for _, row in df.iterrows():
        documents.append({
            'id': row['Activity_ID'],
            'text': row['Key Activities (Missions principales)'],
            'metadata': {
                'entity_code': row['Entity Code'],
                'entity_name': row['Entity Name'],
                'weight_pct': row['%'],
                'excel_row': row['Excel_Row']
            }
        })

    pipeline.vector_store.add(documents)
    count = pipeline.vector_store.count()
    print(f"  Documents vectorized: {count}")

    sample = documents[0]['text'][:80]
    results = pipeline.vector_store.query(sample, top_k=3)
    print(f"  Sanity query: '{sample}...'")
    print(f"  Top matches returned: {len(results)}")

    return documents


def step_upload_to_graph_store(pipeline, graph):
    """Step 4: Upload graph to FalkorDB."""
    print("=" * 60)
    print("STEP 4: UPLOAD TO GRAPH STORE")
    print("=" * 60)

    pipeline.graph_store.upload(graph)

    node_count = pipeline.graph_store.count_nodes()
    edge_count = pipeline.graph_store.count_edges()
    print(f"  Nodes in store: {node_count}")
    print(f"  Edges in store: {edge_count}")

    return {"nodes": node_count, "edges": edge_count}


def step_save_locally(pipeline, graph):
    """Step 5: Save graph locally."""
    print("=" * 60)
    print("STEP 5: SAVE LOCALLY")
    print("=" * 60)

    output_path = pipeline.save(graph)
    print(f"  Saved to: {output_path}")

    return output_path

In [10]:
# Cell 3
data = step_load_data(DATA_SOURCE_PATH)

# Cell 4
graph = step_build_org_graph(data)

# Cell 5 — Inspect
print("--- Org Units and their Activities ---")
for node, attrs in sorted(graph.nodes(data=True)):
    if attrs['type'] == 'org_unit':
        children = list(graph.successors(node))
        activities = [c for c in children if graph.nodes[c]['type'] == 'activity']
        sub_orgs = [c for c in children if graph.nodes[c]['type'] == 'org_unit']
        print(f"  {node:10s} | {len(sub_orgs)} sub-orgs | {len(activities)} activities "
              f"| IDs: {[a for a in activities[:3]]}{'...' if len(activities) > 3 else ''}")

# Cell 6
documents = step_vectorize(pipeline, data)

# Cell 7
store_results = step_upload_to_graph_store(pipeline, graph)

# Cell 8
output_path = step_save_locally(pipeline, graph)

STEP 1: LOAD DATA
  Total activities (rows): 78
  Org units: 12
  Sample activity IDs:
    22.1            (22)
    22.2            (22)
    22.3            (22)
    22.4            (22)
    22.5            (22)


,Activity_ID,Entity Code,Key Activities (Missions principales),%
0,22.1,22,Define strategy and ensure the development of ...,25
1,22.2,22,Ensure the proper functioning and administrati...,25
2,22.3,22,Represent the general management within the bo...,15
3,22.4,22,Manage human resources and the general managem...,15
4,22.5,22,Represent general management in interinstituti...,10


STEP 2: BUILD ORG HIERARCHY + ACTIVITY NODES
  Org unit nodes:    13
  Activity nodes:    78
  Total nodes:       91
  Total edges:       90
  Synthesized parents: ['22B00']
--- Org Units and their Activities ---
  22         | 3 sub-orgs | 7 activities | IDs: ['22.1', '22.2', '22.3']...
  22-10      | 0 sub-orgs | 8 activities | IDs: ['22-10.8', '22-10.9', '22-10.10']...
  22A        | 4 sub-orgs | 6 activities | IDs: ['22A.16', '22A.17', '22A.18']...
  22A10      | 0 sub-orgs | 6 activities | IDs: ['22A10.22', '22A10.23', '22A10.24']...
  22A20      | 0 sub-orgs | 6 activities | IDs: ['22A20.28', '22A20.29', '22A20.30']...
  22A30      | 0 sub-orgs | 6 activities | IDs: ['22A30.34', '22A30.35', '22A30.36']...
  22A40      | 0 sub-orgs | 6 activities | IDs: ['22A40.40', '22A40.41', '22A40.42']...
  22B        | 4 sub-orgs | 6 activities | IDs: ['22B.46', '22B.47', '22B.48']...
  22B00      | 1 sub-orgs | 0 activities | IDs: []
  22B0040    | 0 sub-orgs | 5 activities | IDs: ['22B0040.

AttributeError: 'MockVectorStore' object has no attribute 'add'

In [4]:
# =============================================================================
# RUN PIPELINE
# =============================================================================

# Run the complete pipeline:
# 1. Load data from Excel
# 2. Vectorize in vector store
# 3. Build graph from vector store
# 4. Upload to graph store
# 5. Save locally

results = pipeline.run()

print("\n" + "="*60)
print("PIPELINE RESULTS")
print("="*60)
for key, value in results.items():
    print(f"  {key}: {value}")


ORGANIZATIONAL GRAPH PIPELINE
Data source: ./data/samples/CASP_Activities_EN.xlsx
Vector store: mock
Graph store: networkx
Activities as nodes: True
✓ VectorStore: Mock (in-memory)
✓ GraphStore: NetworkX (local)

STEP 1: LOADING DATA
Loading: CASP_Activities_EN.xlsx
  → 12 orgs, 78 activities

Total: 12 orgs, 78 activities

STEP 2: VECTORIZING

VECTORIZING DATA
Vectorized 90 documents
  Org units: 12
  Activities: 78

STEP 3: BUILDING GRAPH FROM VECTOR STORE

BUILDING GRAPH FROM VECTOR STORE
Documents in vector store: 90
Org nodes: 12
Activity nodes: 78

Graph built:
  Nodes: 90
  Edges: 88
  Org roots: 1
  Activities: 78

STEP 4: UPLOADING TO GRAPH STORE

UPLOADING TO GRAPH STORE

✓ Upload complete:
  Nodes: 91
  Edges: 89

STEP 5: SAVING LOCAL
✓ Graph saved: ./graph_data/organization_graph.json

PIPELINE COMPLETE
  status: completed
  org_units: 12
  activities: 78
  documents: 90
  graph_nodes: 90
  graph_edges: 88

PIPELINE RESULTS
  status: completed
  org_units: 12
  activities:

---
## Step 3: Visualize Tree Structure

In [5]:
# =============================================================================
# DISPLAY ORGANIZATIONAL TREE
# =============================================================================

pipeline.print_tree(show_activities=SHOW_ACTIVITIES_IN_TREE)


ORGANIZATIONAL TREE
[ORG_MASTER] European Parliament Organizations
    └── [22] Directorate-General for Cohesion, Agriculture (L0)
        ├── [22-10] Resources Unit (L1)
        ├── [22A] Directorate for Regional Development, Agricul (L1)
        │   ├── [22A10] Policy Department (L2)
        │   ├── [22A20] Secretariat of the Committee on Regional Deve (L2)
        │   ├── [22A30] Secretariat of the Committee on Agriculture a (L2)
        │   └── [22A40] Secretariat of the Committee on Fisheries (L2)
        └── [22B] Directorate for Transport, Employment and Soc (L1)
            ├── [22B10] Policy Department (L2)
            ├── [22B20] Secretariat of the Committee on Transport and (L2)
            └── [22B30] Secretariat of the Committee on Employment an (L2)


In [ ]:
# =============================================================================
# TREE WITH ACTIVITIES (if enabled)
# =============================================================================

if ACTIVITIES_AS_NODES:
    print("Tree with activities (first org unit only):")
    print()
    
    graph = pipeline.get_graph()
    if graph:
        # Show one org unit with its activities
        sample_org = None
        for node_id in graph.nodes():
            if graph.nodes[node_id].get('node_type') == 'organization':
                children = list(graph.successors(node_id))
                act_children = [c for c in children if graph.nodes[c].get('node_type') == 'activity']
                if act_children:
                    sample_org = node_id
                    break
        
        if sample_org:
            data = graph.nodes[sample_org]
            print(f"[{sample_org}] {data.get('name', '')[:50]}")
            
            children = list(graph.successors(sample_org))
            for child_id in sorted(children)[:5]:
                child_data = graph.nodes[child_id]
                if child_data.get('node_type') == 'activity':
                    weight = child_data.get('weight', 0)
                    desc = child_data.get('description', '')[:60]
                    print(f"  ├── 📋 ({weight:2}%) {desc}...")
            
            if len(children) > 5:
                print(f"  └── ... and {len(children) - 5} more activities")

---
## Step 4: Test Semantic Search

In [ ]:
# =============================================================================
# SEMANTIC SEARCH TEST
# =============================================================================

# Test queries
test_queries = [
    "budget financial management",
    "committee secretariat support",
    "policy research studies",
    "transport tourism"
]

print("\n" + "="*60)
print("SEMANTIC SEARCH RESULTS")
print("="*60)

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    print("-" * 50)
    
    results = pipeline.search_similar(query, k=3)
    
    for i, r in enumerate(results, 1):
        metadata = r.get('metadata', {})
        code = metadata.get('entity_code') or metadata.get('activity_id', 'N/A')
        node_type = metadata.get('node_type', 'org')
        score = r.get('score', 0)
        content = r.get('content', '')[:70]
        
        icon = "📋" if node_type == 'activity' else "🏢"
        print(f"  {i}. {icon} [{code}] (score: {score:.3f})")
        print(f"     {content}...")

---
## Step 5: Graph Analysis

In [ ]:
# =============================================================================
# GRAPH ANALYSIS
# =============================================================================

graph = pipeline.get_graph()

if graph:
    print("\n" + "="*60)
    print("GRAPH ANALYSIS")
    print("="*60)
    
    # Node counts by type
    org_nodes = [n for n in graph.nodes() if graph.nodes[n].get('node_type') == 'organization']
    act_nodes = [n for n in graph.nodes() if graph.nodes[n].get('node_type') == 'activity']
    
    print(f"\n📊 Node Counts:")
    print(f"   Total: {graph.number_of_nodes()}")
    print(f"   Organizations: {len(org_nodes)}")
    print(f"   Activities: {len(act_nodes)}")
    
    # Edge counts by type
    child_edges = [(u, v) for u, v, d in graph.edges(data=True) if d.get('edge_type') == 'HAS_CHILD']
    act_edges = [(u, v) for u, v, d in graph.edges(data=True) if d.get('edge_type') == 'HAS_ACTIVITY']
    
    print(f"\n🔗 Edge Counts:")
    print(f"   Total: {graph.number_of_edges()}")
    print(f"   HAS_CHILD: {len(child_edges)}")
    print(f"   HAS_ACTIVITY: {len(act_edges)}")
    
    # Level distribution
    levels = {}
    for n in org_nodes:
        level = graph.nodes[n].get('level', 0)
        levels[level] = levels.get(level, 0) + 1
    
    print(f"\n📈 Organization Levels:")
    for level in sorted(levels.keys()):
        print(f"   Level {level}: {levels[level]} nodes")
    
    # Org leaves (matching targets)
    org_leaves = [n for n in org_nodes if not any(
        graph.nodes[c].get('node_type') == 'organization' 
        for c in graph.successors(n)
    )]
    
    print(f"\n🎯 Org Leaves (matching targets): {len(org_leaves)}")
    for leaf in sorted(org_leaves):
        name = graph.nodes[leaf].get('name', '')[:40]
        act_count = len([c for c in graph.successors(leaf) if graph.nodes[c].get('node_type') == 'activity'])
        print(f"   [{leaf}] {name} - {act_count} activities")

---
## Step 6: Compare Configurations

In [ ]:
# =============================================================================
# COMPARE: ACTIVITIES AS NODES vs ATTRIBUTES
# =============================================================================

print("\n" + "="*60)
print("CONFIGURATION COMPARISON")
print("="*60)

# Test with activities as nodes
config_nodes = PipelineConfig(
    data_source_path=DATA_SOURCE_PATH,
    activities_as_nodes=True,
    vector_store_type="mock",
    graph_store_type="networkx",
    save_local=False,
    verbose=False
)
pipeline_nodes = OrgGraphPipeline(config_nodes)
results_nodes = pipeline_nodes.run()

# Test with activities as attributes
config_attrs = PipelineConfig(
    data_source_path=DATA_SOURCE_PATH,
    activities_as_nodes=False,
    vector_store_type="mock",
    graph_store_type="networkx",
    save_local=False,
    verbose=False
)
pipeline_attrs = OrgGraphPipeline(config_attrs)
results_attrs = pipeline_attrs.run()

print(f"\n{'Metric':<25} {'Activities=Nodes':<20} {'Activities=Attrs':<20}")
print("-" * 65)
print(f"{'Org Units':<25} {results_nodes['org_units']:<20} {results_attrs['org_units']:<20}")
print(f"{'Activities':<25} {results_nodes['activities']:<20} {results_attrs['activities']:<20}")
print(f"{'Documents Vectorized':<25} {results_nodes['documents']:<20} {results_attrs['documents']:<20}")
print(f"{'Graph Nodes':<25} {results_nodes['graph_nodes']:<20} {results_attrs['graph_nodes']:<20}")
print(f"{'Graph Edges':<25} {results_nodes['graph_edges']:<20} {results_attrs['graph_edges']:<20}")

print("\n📌 Key Differences:")
print("   • Activities=Nodes: Each activity is a separate node in the graph")
print("     → Enables granular activity-to-capability matching")
print("     → More documents vectorized = better semantic search")
print("   • Activities=Attrs: Activities stored as attributes on org nodes")
print("     → Simpler graph structure")
print("     → Org-level matching only")

---
## Step 7: Test Load from Local

In [ ]:
# =============================================================================
# TEST: LOAD GRAPH FROM LOCAL JSON
# =============================================================================

import os
from storage_impl import NetworkXGraphStore, StorageConfig

if os.path.exists(GRAPH_LOCAL_PATH):
    print("\n" + "="*60)
    print("LOADING GRAPH FROM LOCAL FILE")
    print("="*60)
    
    # Create new graph store and load
    storage_config = StorageConfig(graph_local_path=GRAPH_LOCAL_PATH)
    graph_store = NetworkXGraphStore(storage_config)
    loaded_graph = graph_store.load(GRAPH_LOCAL_PATH)
    
    # Verify
    stats = graph_store.get_stats()
    print(f"\n✓ Graph loaded:")
    print(f"   Nodes: {stats['nodes']}")
    print(f"   Edges: {stats['edges']}")
    
    # Compare with pipeline graph
    pipeline_graph = pipeline.get_graph()
    if pipeline_graph:
        print(f"\n   Match pipeline graph: {stats['nodes'] == pipeline_graph.number_of_nodes()}")
else:
    print(f"⚠️ Local file not found: {GRAPH_LOCAL_PATH}")

---
## Step 8: Direct Storage Access

In [ ]:
# =============================================================================
# DIRECT ACCESS TO STORAGE COMPONENTS
# =============================================================================

print("\n" + "="*60)
print("STORAGE COMPONENT ACCESS")
print("="*60)

# Access vector store
if pipeline.vector_store:
    print(f"\n📦 Vector Store:")
    print(f"   Type: {type(pipeline.vector_store).__name__}")
    print(f"   Document count: {pipeline.vector_store.count()}")

# Access graph store
if pipeline.graph_store:
    print(f"\n🔗 Graph Store:")
    print(f"   Type: {type(pipeline.graph_store).__name__}")
    stats = pipeline.graph_store.get_stats()
    for key, value in stats.items():
        print(f"   {key}: {value}")

# Access graph builder
if pipeline.graph_builder:
    print(f"\n🏗️ Graph Builder:")
    print(f"   Org roots: {pipeline.graph_builder.org_roots}")
    print(f"   Activity nodes: {len(pipeline.graph_builder.activity_nodes)}")
    print(f"   Embeddings computed: {len(pipeline.graph_builder.embeddings)}")

---
## Summary

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*70)
print("PIPELINE EXECUTION SUMMARY")
print("="*70)

print(f"\n📁 Data Source:")
print(f"   Path: {DATA_SOURCE_PATH}")
print(f"   Activities as nodes: {ACTIVITIES_AS_NODES}")

print(f"\n📦 Vector Store:")
print(f"   Type: {VECTOR_STORE_TYPE}")
print(f"   Documents: {results.get('documents', 0)}")

print(f"\n🔗 Graph Store:")
print(f"   Type: {'falkordb' if USE_FALKORDB else GRAPH_STORE_TYPE}")
print(f"   Nodes: {results.get('graph_nodes', 0)}")
print(f"   Edges: {results.get('graph_edges', 0)}")

print(f"\n💾 Outputs:")
if SAVE_LOCAL:
    print(f"   Local graph: {GRAPH_LOCAL_PATH}")
if USE_FALKORDB:
    print(f"   FalkorDB graph: {FALKORDB_GRAPH_NAME}")

print(f"\n✅ Status: {results.get('status', 'unknown')}")